In [1]:
import pandas as pd
import polars as pl
import util
from pathlib import Path

pd.set_option('display.float_format', '{:,.0f}'.format)

## Average Miles Walking and Biking per Day by Resident

In [2]:
# trip,person,vmt data
person_trips = pd.read_csv(util.output_path / 'agg/dash/person_trips.csv')
person = pd.read_csv(util.output_path / 'agg/dash/person_geog.csv')
vmt = pd.read_csv(util.output_path / 'agg/dash/person_vmt.csv')

# list of equity geographies
equity_geogs = util.summary_config['hh_equity_geogs']

# TRIPS
df_trip = person_trips.copy()

# add home RGC
df_trip = util.get_is_rgc(df_trip, 'hh_rgc')
# add trip type
df_trip.loc[df_trip['dpurp'] != 'Work', 'trip_type'] = 'Non-Work'
df_trip.loc[df_trip['dpurp'] == 'Work', 'trip_type'] = 'Work'

# PERSONS
df_person = person.copy()
# add home RGC
df_person = util.get_is_rgc(df_person, 'hh_rgc')

# VMT
df_vmt = vmt.copy()
# add home RGC
df_vmt = util.get_is_rgc(df_vmt, 'hh_rgc')

# Select only walk and bike trips
df_vmt_bp = df_vmt[df_vmt['mode'].isin(['Walk','Bike'])].copy()
# Select only drivers (dorp = 1) and auto trips
df_vmt = df_vmt[df_vmt['mode'].isin(['SOV','HOV2','HOV3+']) & (df_vmt['dorp'] == 1)].copy()

# total population by equity geography
equity_geogs_population = df_person[equity_geogs].apply(lambda x: x * df_person['psexpfac']).sum().reset_index()
equity_geogs_population.columns = ['Equity Group', 'psexpfac']

df_vmt_bp2 = df_vmt_bp.copy()
df_vmt_bp2['mode'] = 'Walk and Bike'
# add walk and bike
df_vmt_bp2 = pd.concat([df_vmt_bp, df_vmt_bp2])



In [3]:
def walk_bike_per_person(geog, map=False):
    
    # miles by mode and geography
    df1 = df_vmt_bp2.groupby([geog, 'mode'], as_index=False)['travdist_wt'].sum().set_index(geog)

    if map:
        df1.index = df1.index.astype('int').map({
                                0: 'Below Regional Average', 
                                1: 'Above Regional Average', 
                                2: 'Higher Share of Equity Population',
                                })

    # add total miles in region
    df1_region = df_vmt_bp2.groupby(['mode'])['travdist_wt'].sum().reset_index()
    df1_region[geog] = 'Region'
    df1_region = df1_region.set_index(geog)
    df1 = pd.concat([df1, df1_region])

    # population by geography
    df3 = df_person.groupby(geog, as_index=False)['psexpfac'].sum().set_index(geog)
    if map:
        df3.index = df3.index.astype('int').map({
                                0: 'Below Regional Average', 
                                1: 'Above Regional Average', 
                                2: 'Higher Share of Equity Population',
                                })
    df3.loc['Region',:] = df3.sum(axis=0)

    # calculate average miles per person
    df = df1.merge(df3, on=geog)
    df['Average Miles per Person'] = df['travdist_wt']/df['psexpfac']

    return df.pivot(columns='mode', values='Average Miles per Person')

In [4]:
pd.set_option('display.float_format', '{:,.2f}'.format)
walk_bike_per_person('hh_county')

mode,Bike,Walk,Walk and Bike
hh_county,,,
King,0.22,0.57,0.79
Kitsap,0.14,0.51,0.66
Pierce,0.17,0.48,0.64
Region,0.19,0.53,0.72
Snohomish,0.16,0.46,0.62


In [5]:
walk_bike_per_person('is_rgc')

mode,Bike,Walk,Walk and Bike
is_rgc,,,
In RGC,0.27,1.05,1.32
Not in RGC,0.18,0.49,0.68
Region,0.19,0.53,0.72


In [6]:
walk_bike_per_person('hh_rgc')

mode,Bike,Walk,Walk and Bike
hh_rgc,,,
Auburn,0.36,0.69,1.05
Bellevue,0.29,0.86,1.15
Bothell Canyon Park,0.17,0.48,0.65
Bremerton,0.21,0.92,1.13
Burien,0.27,0.55,0.82
Everett,0.14,0.73,0.87
Federal Way,0.25,0.61,0.86
Greater Downtown Kirkland,0.28,0.59,0.86
Kent,0.26,0.67,0.94


In [7]:
walk_bike_per_person('hh_rg_proposed')

mode,Bike,Walk,Walk and Bike
hh_rg_proposed,,,
Cities and Towns,0.14,0.43,0.58
Core Cities,0.20,0.50,0.70
High Capacity Transit Communities,0.17,0.48,0.65
Metropolitan Cities,0.23,0.72,0.95
Region,0.19,0.53,0.72
Rural Areas,0.16,0.32,0.48
Urban Unincorporated Areas,0.15,0.44,0.59


In [8]:
efa_col_dict = {
    "People of Color": "hh_efa_poc",
    "Income": "hh_efa_pov200",
    "LEP": "hh_efa_lep",
    "Disability": "hh_efa_dis",
    "Older Adults": "hh_efa_older",
    "Youth": "hh_efa_youth"
    }

In [9]:
df = pd.DataFrame()
for label, col in efa_col_dict.items():
    _df = walk_bike_per_person(col, map=True)
    _df = _df.drop('Region', axis=0)
    _df['Group'] = label
    df = pd.concat([df, _df])
df = df.reset_index()
df.rename(columns={'index': 'EFA Type'}, inplace=True)
df

mode,EFA Type,Bike,Walk,Walk and Bike,Group
0,Above Regional Average,0.20,0.56,0.76,People of Color
1,Below Regional Average,0.18,0.50,0.68,People of Color
2,Higher Share of Equity Population,0.22,0.56,0.77,People of Color
3,Above Regional Average,0.19,0.55,0.73,Income
4,Below Regional Average,0.18,0.50,0.69,Income
5,Higher Share of Equity Population,0.22,0.61,0.82,Income
6,Above Regional Average,0.19,0.52,0.71,LEP
7,Below Regional Average,0.18,0.53,0.71,LEP
8,Higher Share of Equity Population,0.21,0.53,0.75,LEP
9,Above Regional Average,0.19,0.53,0.71,Disability


## Median Trip Distance by Mode and Home Location 

In [10]:
trip = pl.read_csv(util.output_path / 'daysim/_trip.tsv',separator='\t')
hh = pl.read_csv(util.output_path / 'daysim/_household.tsv',separator='\t')

list_cols = ['ParcelID','CountyName','GrowthCenterName','rg_proposed', 'People of Color', 'Income', 'LEP', 'Disability',
       'Older Adults', 'Youth']
parcel_geog = util.get_parcel_geog()[list_cols]


In [11]:
# trip table with home location
df_trip_hh = trip.join(hh, on="hhno", how="left").to_pandas()
df_trip_hh = df_trip_hh.merge(parcel_geog, how='left', left_on='hhparcel', right_on='ParcelID')
df_trip_hh['mode_label'] = df_trip_hh['mode'].map(
    {1: 'Walk',
     2: 'Bike',
     3: 'Drive Alone',
     4: 'Shared Ride',
     5: 'Shared Ride',
     6: 'Transit'}
     )

df_trip_hh = df_trip_hh[df_trip_hh['CountyName']!='Outside Region']
df_trip_hh = util.get_is_rgc(df_trip_hh, 'GrowthCenterName', 'is_rgc')


In [12]:
def get_median_time_distance(df, geog, value_col):

    df_geog = pd.pivot_table(df, index=geog, columns='mode_label', aggfunc='median', values=value_col)
    
    return df_geog


df = get_median_time_distance(df_trip_hh,'CountyName','travdist')
df_region =  get_median_time_distance(df_trip_hh,None,'travdist')
df.loc['Region'] = df_region.iloc[0]
df

mode_label,Bike,Drive Alone,Shared Ride,Transit,Walk
CountyName,,,,,
King,1.66,3.58,2.70,4.18,0.65
Kitsap,1.66,3.08,2.48,13.75,0.72
Pierce,1.81,3.80,2.82,4.57,0.81
Snohomish,1.79,3.82,2.66,4.08,0.82
Region,1.72,3.63,2.70,4.42,0.70


In [13]:
df = get_median_time_distance(df_trip_hh,'rg_proposed','travdist')
df

mode_label,Bike,Drive Alone,Shared Ride,Transit,Walk
rg_proposed,,,,,
Cities and Towns,1.55,4.42,2.61,8.51,0.76
Core Cities,1.85,3.47,2.56,6.67,0.81
High Capacity Transit Communities,1.87,3.75,2.68,8.07,0.85
Metropolitan Cities,1.45,2.53,1.99,3.37,0.59
Rural Areas,2.96,6.41,5.28,15.53,0.94
Urban Unincorporated Areas,1.96,4.30,3.07,10.34,1.01


In [14]:
df = get_median_time_distance(df_trip_hh,'is_rgc','travdist')
df

mode_label,Bike,Drive Alone,Shared Ride,Transit,Walk
is_rgc,,,,,
In RGC,1.04,1.71,1.41,1.66,0.46
Not in RGC,1.87,3.73,2.75,5.24,0.79


In [15]:
df = get_median_time_distance(df_trip_hh,'GrowthCenterName','travdist')
df

mode_label,Bike,Drive Alone,Shared Ride,Transit,Walk
GrowthCenterName,,,,,
Auburn,1.52,2.38,1.66,9.93,0.43
Bellevue,0.94,1.39,1.02,5.51,0.35
Bothell Canyon Park,2.04,3.40,2.45,14.00,0.65
Bremerton,1.32,1.54,1.14,22.76,0.43
Burien,1.91,4.30,2.60,8.09,0.41
Everett,0.86,1.18,0.89,1.24,0.40
Federal Way,1.69,2.28,1.69,6.08,0.50
Greater Downtown Kirkland,2.20,3.40,2.37,8.83,0.52
Kent,1.45,2.31,2.03,7.14,0.40


In [16]:
df = pd.DataFrame()
for col in efa_col_dict.keys():
    _df = get_median_time_distance(df_trip_hh,col,'travdist').reset_index()
    _df['Group'] = col
    _df.rename(columns={col: 'EFA Type'}, inplace=True)
    df = pd.concat([df, _df])

df.set_index(['Group','EFA Type'])

mode_label                                         Bike  Drive Alone  \
Group           EFA Type                                               
People of Color Above Regional Average             1.62         3.42   
                Below Regional Average             1.78         3.82   
                Higher Share of Equity Population  1.72         3.50   
Income          Above Regional Average             1.65         3.34   
                Below Regional Average             1.78         3.90   
                Higher Share of Equity Population  1.60         2.96   
LEP             Above Regional Average             1.77         3.74   
                Below Regional Average             1.67         3.67   
                Higher Share of Equity Population  1.82         3.39   
Disability      Above Regional Average             1.67         3.55   
                Below Regional Average             1.78         3.77   
                Higher Share of Equity Population  1.56         3.18   
Older Adults    Above Regional Average             1.79         3.82   
                Below Regional Average             1.66         3.47   
                Higher Share of Equity Population  1.82         3.88   
Youth           Above Regional Average             1.90         3.99   
                Below Regional Average             1.59         3.17   
                Higher Share of Equity Population  1.86         4.54   

mode_label                                         Shared Ride  Transit  Walk  
Group           EFA Type                                                       
People of Color Above Regional Average                    2.53     3.45  0.68  
                Below Regional Average                    2.84     5.02  0.72  
                Higher Share of Equity Population         2.62     4.65  0.70  
Income          Above Regional Average                    2.49     4.48  0.69  
                Below Regional Average                    2.90     4.72  0.71  
                Higher Share of Equity Population         2.33     3.91  0.69  
LEP             Above Regional Average                    2.74     4.33  0.72  
                Below Regional Average                    2.76     4.25  0.68  
                Higher Share of Equity Population         2.51     5.25  0.78  
Disability      Above Regional Average                    2.63     4.80  0.70  
                Below Regional Average                    2.80     4.48  0.73  
                Higher Share of Equity Population         2.44     3.01  0.65  
Older Adults    Above Regional Average                    2.86     4.66  0.71  
                Below Regional Average                    2.55     4.02  0.70  
                Higher Share of Equity Population         3.02     6.68  0.71  
Youth           Above Regional Average                    2.91     6.11  0.83  
                Below Regional Average                    2.43     3.90  0.63  
                Higher Share of Equity Population         3.05     6.94  0.90

## Median Time Spent Walking/Biking
Time in minutes

In [17]:

df = get_median_time_distance(df_trip_hh,'CountyName','travtime')
df_region =  get_median_time_distance(df_trip_hh,None,'travtime')
df.loc['Region'] = df_region.iloc[0]
df

mode_label,Bike,Drive Alone,Shared Ride,Transit,Walk
CountyName,,,,,
King,11.10,12.39,10.34,24.87,12.98
Kitsap,11.07,10.72,9.55,75.00,14.34
Pierce,12.04,12.36,10.51,38.82,16.23
Snohomish,11.90,11.71,9.41,30.65,16.41
Region,11.44,12.14,10.15,27.05,14.09


In [18]:
df = get_median_time_distance(df_trip_hh,'rg_proposed','travtime')
df

mode_label,Bike,Drive Alone,Shared Ride,Transit,Walk
rg_proposed,,,,,
Cities and Towns,10.34,12.37,9.34,44.87,15.17
Core Cities,12.35,11.28,9.47,39.07,16.28
High Capacity Transit Communities,12.49,11.75,9.61,41.55,17.04
Metropolitan Cities,9.69,11.51,9.87,21.28,11.80
Rural Areas,19.75,15.93,14.30,63.66,18.75
Urban Unincorporated Areas,13.09,12.39,10.28,52.38,20.12


In [19]:
df = get_median_time_distance(df_trip_hh,'is_rgc','travtime')
df

mode_label,Bike,Drive Alone,Shared Ride,Transit,Walk
is_rgc,,,,,
In RGC,6.90,10.65,9.46,15.31,9.11
Not in RGC,12.47,12.24,10.17,30.76,15.81


In [20]:
df = get_median_time_distance(df_trip_hh,'GrowthCenterName','travtime')
df

mode_label,Bike,Drive Alone,Shared Ride,Transit,Walk
GrowthCenterName,,,,,
Auburn,10.15,9.88,8.28,46.78,8.60
Bellevue,6.26,8.16,7.24,27.81,7.08
Bothell Canyon Park,13.61,11.18,9.73,51.83,12.92
Bremerton,8.78,9.98,9.37,65.59,8.56
Burien,12.71,11.64,9.00,40.55,8.28
Everett,5.73,7.53,6.79,14.99,7.93
Federal Way,11.25,8.87,7.62,39.94,10.03
Greater Downtown Kirkland,14.65,10.85,9.13,37.43,10.32
Kent,9.66,8.48,7.48,40.37,8.06


In [21]:
df = pd.DataFrame()
for col in efa_col_dict.keys():
    _df = get_median_time_distance(df_trip_hh,col,'travtime').reset_index()
    _df['Group'] = col
    _df.rename(columns={col: 'EFA Type'}, inplace=True)
    df = pd.concat([df, _df])

df.set_index(['Group','EFA Type'])

mode_label                                         Bike  Drive Alone  \
Group           EFA Type                                               
People of Color Above Regional Average            10.82        11.66   
                Below Regional Average            11.85        12.54   
                Higher Share of Equity Population 11.46        11.78   
Income          Above Regional Average            11.01        11.62   
                Below Regional Average            11.88        12.59   
                Higher Share of Equity Population 10.68        11.10   
LEP             Above Regional Average            11.77        11.98   
                Below Regional Average            11.16        12.40   
                Higher Share of Equity Population 12.12        11.45   
Disability      Above Regional Average            11.16        11.94   
                Below Regional Average            11.89        12.42   
                Higher Share of Equity Population 10.41        11.45   
Older Adults    Above Regional Average            11.93        12.46   
                Below Regional Average            11.10        11.86   
                Higher Share of Equity Population 12.15        12.39   
Youth           Above Regional Average            12.69        12.36   
                Below Regional Average            10.60        11.80   
                Higher Share of Equity Population 12.42        12.98   

mode_label                                         Shared Ride  Transit  Walk  
Group           EFA Type                                                       
People of Color Above Regional Average                    9.73    23.86 13.70  
                Below Regional Average                   10.48    27.78 14.38  
                Higher Share of Equity Population         9.94    29.87 13.96  
Income          Above Regional Average                    9.77    27.58 13.88  
                Below Regional Average                   10.45    27.80 14.27  
                Higher Share of Equity Population         9.62    24.48 13.85  
LEP             Above Regional Average                    9.96    28.74 14.40  
                Below Regional Average                   10.41    24.96 13.62  
                Higher Share of Equity Population         9.58    31.82 15.53  
Disability      Above Regional Average                   10.01    29.68 14.07  
                Below Regional Average                   10.28    26.32 14.53  
                Higher Share of Equity Population         9.88    24.85 12.90  
Older Adults    Above Regional Average                   10.52    29.82 14.19  
                Below Regional Average                    9.79    24.46 14.01  
                Higher Share of Equity Population        10.78    34.80 14.15  
Youth           Above Regional Average                   10.20    35.89 16.51  
                Below Regional Average                   10.03    23.68 12.68  
                Higher Share of Equity Population        10.35    40.56 17.95

## % Population Walking or Biking for Transportation
Does not include people making exercise trips

In [22]:
bike_walk_trips = trip.filter(pl.col("mode").is_in([1, 2]))

# Get unique persons with at least one bike/walk trip
bike_walk_persons = bike_walk_trips.select(["hhno", "pno"]).unique()
bike_walk_persons = bike_walk_persons.with_columns(bike_walk=pl.lit(1))

# Join back to all persons, mark bike_walk as False if not present
person = pl.read_csv(util.output_path / 'daysim/_person.tsv',separator='\t')
person_with_bike_walk = person.join(bike_walk_persons, on=["hhno", "pno"], how="left")
person_with_bike_walk = person_with_bike_walk.join(hh, on='hhno', how='left')

person_with_bike_walk = person_with_bike_walk.to_pandas()
person_with_bike_walk = person_with_bike_walk.merge(parcel_geog, how='left', left_on='hhparcel', right_on='ParcelID')

person_with_bike_walk['bike_walk'] = person_with_bike_walk['bike_walk'].fillna(0)


person_with_bike_walk = person_with_bike_walk[person_with_bike_walk['CountyName']!='Outside Region']
person_with_bike_walk = util.get_is_rgc(person_with_bike_walk, 'GrowthCenterName', 'is_rgc')

In [23]:
def get_pop_bike_walk(df, geog):
    df = pd.pivot_table(df, index=geog, columns='bike_walk', aggfunc='sum', values='psexpfac')
    df['% Biking or Walking'] = df[1]/(df[0]+df[1])
    return df[['% Biking or Walking']]

In [24]:
pd.set_option('display.float_format', '{:.1%}'.format)
df = get_pop_bike_walk(person_with_bike_walk, 'CountyName')
df_region =  get_pop_bike_walk(person_with_bike_walk, None)
df.loc['Region'] = df_region.iloc[0]
df

bike_walk,% Biking or Walking
CountyName,
King,28.8%
Kitsap,24.9%
Pierce,21.8%
Snohomish,21.8%
Region,25.7%


In [25]:
df = get_pop_bike_walk(person_with_bike_walk, 'rg_proposed')
df

bike_walk,% Biking or Walking
rg_proposed,
Cities and Towns,22.3%
Core Cities,23.6%
High Capacity Transit Communities,22.2%
Metropolitan Cities,37.4%
Rural Areas,12.9%
Urban Unincorporated Areas,18.7%


In [26]:
df = get_pop_bike_walk(person_with_bike_walk, 'is_rgc')
df

bike_walk,% Biking or Walking
is_rgc,
In RGC,57.2%
Not in RGC,23.5%


In [27]:
df = get_pop_bike_walk(person_with_bike_walk, 'GrowthCenterName')
df

bike_walk,% Biking or Walking
GrowthCenterName,
Auburn,40.6%
Bellevue,59.2%
Bothell Canyon Park,23.7%
Bremerton,53.1%
Burien,36.6%
Everett,50.2%
Federal Way,35.6%
Greater Downtown Kirkland,33.6%
Kent,40.5%


In [28]:
df = pd.DataFrame()
for col in efa_col_dict.keys():
    _df = get_pop_bike_walk(person_with_bike_walk,col).reset_index()
    _df['Group'] = col
    _df.rename(columns={col: 'EFA Type'}, inplace=True)
    df = pd.concat([df, _df])

df.set_index(['Group','EFA Type'])

bike_walk                                          % Biking or Walking
Group           EFA Type                                              
People of Color Above Regional Average                           27.2%
                Below Regional Average                           24.4%
                Higher Share of Equity Population                27.0%
Income          Above Regional Average                           26.7%
                Below Regional Average                           24.4%
                Higher Share of Equity Population                29.4%
LEP             Above Regional Average                           24.8%
                Below Regional Average                           26.0%
                Higher Share of Equity Population                25.7%
Disability      Above Regional Average                           25.6%
                Below Regional Average                           25.0%
                Higher Share of Equity Population                28.2%
Older Adults    Above Regional Average                           24.1%
                Below Regional Average                           27.2%
                Higher Share of Equity Population                23.8%
Youth           Above Regional Average                           21.7%
                Below Regional Average                           30.1%
                Higher Share of Equity Population                20.1%

## Bike and Walk Tours by Purpose
Tours where most or all trips are by the mode.

In [29]:
df = pd.read_csv(util.output_path / 'agg/dash/tour_total.csv')

df['Tour Mode'] = df['tmodetp'].map({'SOV': 'Drove Alone',
                                     'HOV2': 'Shared Ride',
                                     'HOV3+': 'Shared Ride',
                                     'Transit': 'Transit',
                                     'Walk': 'Walk',
                                     'Bike': 'Bike'
                                     })

df = pd.pivot_table(df, index='pdpurp', columns='Tour Mode', values='toexpfac', aggfunc='sum')
df = df/df.sum()
df[['Transit', 'Walk', 'Bike','Drove Alone', 'Shared Ride', ]]

Tour Mode,Transit,Walk,Bike,Drove Alone,Shared Ride
pdpurp,,,,,
Escort,0.0%,6.9%,2.5%,1.0%,22.7%
Meal,8.3%,10.0%,1.7%,4.7%,7.4%
Personal Business,12.7%,5.6%,4.2%,8.3%,5.9%
School,19.4%,4.1%,16.0%,4.0%,11.7%
Shop,16.0%,11.8%,7.7%,15.2%,10.9%
Social,18.3%,51.7%,39.0%,20.7%,22.5%
Work,25.4%,9.9%,28.9%,46.1%,18.9%


## Average Jobs Accessible within 1 Mile Walk and 3 Mile Bike
Note that this is not using the bike network, but is instead using the all-streets network.

Average accessible jobs are weighted averages based on parcel household population.

In [30]:
# total jobs
parcel_emp = util.get_parcels_urbansim_data()[['parcelid','emptot_p']]
tot_jobs = parcel_emp['emptot_p'].sum()

In [31]:
walk_bike_jobs_access = pd.read_csv(util.output_path / 'access/walk_bike_jobs_access.csv', 
                                  usecols=['geography_value', 'jobs_1_mile_walk', 'jobs_3_mile_bike', 'geography_group']).\
                                  rename(columns={'geography_value': 'geography'})

df_access = walk_bike_jobs_access.copy()
df_access = df_access[df_access['geography'] != 'Outside Region']
# rename region
df_access.loc[df_access['geography_group'] == 'region', 'geography'] = 'Region'
# rename rgc
df_access.loc[df_access['geography_group'] == 'rgc_binary', 'geography'] = ['Not in RGC', 'In RGC']
# rename efa 
# jobs access in equity geographies
equity_geogs = util.summary_config['equity_geogs']
df_access.loc[df_access['geography_group'].isin(equity_geogs), 'geography'] = df_access.loc[df_access['geography_group'].isin(equity_geogs), 'geography'].\
    map({"0.0": 'Below Regional Average', 
         "1.0": 'Above Regional Average', 
         "2.0": 'Higher Share of Equity Population'}
         )

In [32]:
def bp_job_access_geog(access_table,geog):
    df = access_table.loc[access_table['geography_group'].isin(geog)].\
        rename(columns={'jobs_1_mile_walk': 'Jobs within 1-mile Walk',
                        'jobs_3_mile_bike': 'Jobs within 3-mile Bike'}).\
        drop(columns=['geography_group']).\
        set_index('geography')

    df['% Total Jobs (1-mile Walk)'] = df['Jobs within 1-mile Walk'].apply(lambda x: x / tot_jobs)
    df['% Total Jobs (3-mile Bike)'] = df['Jobs within 3-mile Bike'].apply(lambda x: x / tot_jobs)

    

    return df

format_dict = {
        'Jobs within 1-mile Walk': "{:.0f}",
        'Jobs within 3-mile Bike': "{:.0f}", 
        '% Total Jobs (1-mile Walk)': "{:.1%}", 
        '% Total Jobs (3-mile Bike)': "{:.1%}"
    }

In [33]:
df = bp_job_access_geog(df_access,['region','CountyName'])
df.style.format(format_dict)

,Jobs within 1-mile Walk,Jobs within 3-mile Bike,% Total Jobs (1-mile Walk),% Total Jobs (3-mile Bike)
geography,,,,
King,18516,84913,0.9%,3.9%
Kitsap,1236,8014,0.1%,0.4%
Pierce,2508,18163,0.1%,0.8%
Snohomish,2006,17703,0.1%,0.8%
Region,11164,54254,0.5%,2.5%


In [34]:
df = bp_job_access_geog(df_access,['rgc_binary'])
df.style.format(format_dict)

,Jobs within 1-mile Walk,Jobs within 3-mile Bike,% Total Jobs (1-mile Walk),% Total Jobs (3-mile Bike)
geography,,,,
Not in RGC,2912,34868,0.1%,1.6%
In RGC,88878,236824,4.1%,11.0%


In [35]:
df = bp_job_access_geog(df_access,['GrowthCenterName'])
df.style.format(format_dict)

,Jobs within 1-mile Walk,Jobs within 3-mile Bike,% Total Jobs (1-mile Walk),% Total Jobs (3-mile Bike)
geography,,,,
Auburn,10541,40324,0.5%,1.9%
Bellevue,59188,110699,2.7%,5.1%
Bothell Canyon Park,8539,21748,0.4%,1.0%
Bremerton,11536,34387,0.5%,1.6%
Burien,4829,13400,0.2%,0.6%
Everett,15908,39722,0.7%,1.8%
Federal Way,6348,26221,0.3%,1.2%
Greater Downtown Kirkland,12208,36556,0.6%,1.7%
Issaquah,nan,nan,nan%,nan%


In [36]:
df = bp_job_access_geog(df_access,['rg_proposed'])
df.style.format(format_dict)

,Jobs within 1-mile Walk,Jobs within 3-mile Bike,% Total Jobs (1-mile Walk),% Total Jobs (3-mile Bike)
geography,,,,
Cities and Towns,984,8948,0.0%,0.4%
Core Cities,3415,28587,0.2%,1.3%
High Capacity Transit Communities,1428,15237,0.1%,0.7%
Metropolitan Cities,29432,128074,1.4%,5.9%
Rural Areas,142,2387,0.0%,0.1%
Urban Unincorporated Areas,443,6816,0.0%,0.3%


In [37]:
efa_geog_dict = {
    "People of Color": "equity_focus_areas_2023__efa_poc",
    "Income": "equity_focus_areas_2023__efa_pov200",
    "LEP": "equity_focus_areas_2023__efa_lep",
    "Disability": "equity_focus_areas_2023__efa_dis",
    "Older Adults": "equity_focus_areas_2023__efa_older",
    "Youth": "equity_focus_areas_2023__efa_youth"
    }

df = pd.DataFrame()
for label, col in efa_geog_dict.items():
    _df = bp_job_access_geog(df_access,[col])
    _df['Group'] = label
    df = pd.concat([df, _df])
df = df.reset_index()
df.rename(columns={'geography': 'EFA Type'}, inplace=True)
df.set_index(['Group','EFA Type']).style.format(format_dict)